# Results paragraph: Batch, Age, and Quality (Figure 7)

Computes Figure 7 GE-specific age-IQM GAM associations, batch mediation, and FA before/after-QC correspondence (`rho`) from:
- `data/harmonized_data/merged_data_meisler_analyses_harmonized.parquet`
- `data/age_effects/age_effects_all_outputs.rds`


In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(arrow)
  library(fs)
  library(jsonlite)
  library(mgcv)
  library(purrr)
})

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) stop("Could not locate config.json.")
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)

harm_path <- fs::path(project_root, "data", "harmonized_data", "merged_data_meisler_analyses_harmonized.parquet")
age_effect_file <- fs::path(project_root, "data", "age_effects", "age_effects_all_outputs.rds")
if (!file.exists(harm_path)) stop("Missing harmonized parquet: ", harm_path)
if (!file.exists(age_effect_file)) stop("Missing age-effects file: ", age_effect_file)

iqms_ge_grid <- c("t1post_dwi_contrast", "mean_fd", "qc_prediction", "t1post_neighbor_corr")
iqm_label_map <- c(
  t1post_dwi_contrast = "Preprocessed dMRI Contrast",
  mean_fd = "Mean Framewise Displacement",
  qc_prediction = "Quality Classifier Score",
  t1post_neighbor_corr = "Preprocessed NDC"
)

fmt_pct1 <- function(x) sprintf("%.1f", 100 * x)
clamp01 <- function(x) pmin(pmax(x, 0), 1)


In [ ]:
# --- GE-specific age-IQM and batch mediation stats (same logic as Figure 7 row 1) ---
need_harm <- c("age", "scanner_manufacturer", "batch_device_software", iqms_ge_grid)
df_ge <- arrow::read_parquet(harm_path, col_select = tidyselect::all_of(need_harm)) %>%
  as_tibble() %>%
  mutate(
    age = as.numeric(age),
    scanner_manufacturer = as.character(scanner_manufacturer),
    batch_device_software = as.character(batch_device_software)
  ) %>%
  filter(scanner_manufacturer == "GE", !is.na(age), !is.na(batch_device_software))

ge_gam_stats <- purrr::map_dfr(iqms_ge_grid, function(iqm_col) {
  df_iqm <- df_ge %>%
    transmute(
      age = age,
      batch_device_software = as.factor(batch_device_software),
      iqm_val = suppressWarnings(as.numeric(.data[[iqm_col]]))
    ) %>%
    filter(complete.cases(.))

  if (nrow(df_iqm) < 50) {
    return(tibble(iqm = iqm_col, n = nrow(df_iqm), r2_age_only = NA_real_, prop_mediated_gam = NA_real_, r2_batch = NA_real_, r2_age_given_batch = NA_real_))
  }

  m_age_only <- tryCatch(mgcv::gam(iqm_val ~ s(age, k = 4), data = df_iqm, method = "REML"), error = function(e) NULL)
  m_batch_only <- tryCatch(mgcv::gam(iqm_val ~ batch_device_software, data = df_iqm, method = "REML"), error = function(e) NULL)
  m_full <- tryCatch(mgcv::gam(iqm_val ~ s(age, k = 4) + batch_device_software, data = df_iqm, method = "REML"), error = function(e) NULL)

  if (is.null(m_age_only) || is.null(m_batch_only) || is.null(m_full)) {
    return(tibble(iqm = iqm_col, n = nrow(df_iqm), r2_age_only = NA_real_, prop_mediated_gam = NA_real_, r2_batch = NA_real_, r2_age_given_batch = NA_real_))
  }

  r2_total <- max(0, summary(m_age_only)$r.sq)
  r2_batch <- max(0, summary(m_batch_only)$r.sq)
  r2_full <- max(0, summary(m_full)$r.sq)
  r2_age_given_batch <- max(0, r2_full - r2_batch)
  prop_mediated <- if (r2_total < 1e-10) NA_real_ else clamp01(1 - (r2_age_given_batch / r2_total))

  tibble(
    iqm = iqm_col,
    n = nrow(df_iqm),
    r2_age_only = r2_total,
    prop_mediated_gam = prop_mediated,
    r2_batch = r2_batch,
    r2_age_given_batch = r2_age_given_batch
  )
}) %>%
  mutate(iqm_label = unname(iqm_label_map[iqm]))

# pull focal values for paragraph
r_mean_fd <- ge_gam_stats %>% filter(iqm == "mean_fd") %>% pull(r2_age_only)
r_qc <- ge_gam_stats %>% filter(iqm == "qc_prediction") %>% pull(r2_age_only)
r_ndc <- ge_gam_stats %>% filter(iqm == "t1post_neighbor_corr") %>% pull(r2_age_only)
r_contrast <- ge_gam_stats %>% filter(iqm == "t1post_dwi_contrast") %>% pull(r2_age_only)

med_non_contrast <- ge_gam_stats %>%
  filter(iqm %in% c("mean_fd", "qc_prediction", "t1post_neighbor_corr")) %>%
  pull(prop_mediated_gam)
med_non_contrast_min <- min(med_non_contrast, na.rm = TRUE)
med_contrast <- ge_gam_stats %>% filter(iqm == "t1post_dwi_contrast") %>% pull(prop_mediated_gam)

# --- FA developmental effects with vs without IQM covariates (same logic as Figure 7 row 3) ---
df_age_all <- readRDS(age_effect_file)
row3_metric <- "GQI_fa"

no_qc <- df_age_all %>%
  filter(
    metric == row3_metric,
    source == "harmonized",
    output_type == "vendorwise",
    scanner_manufacturer == "GE",
    qc_metric == "no_quality",
    !is.na(bundle),
    !is.na(age_effect_size)
  ) %>%
  select(bundle, age_no_qc = age_effect_size)

with_qc <- df_age_all %>%
  filter(
    metric == row3_metric,
    source == "harmonized",
    output_type == "vendorwise",
    scanner_manufacturer == "GE",
    qc_metric %in% iqms_ge_grid,
    !is.na(bundle),
    !is.na(age_effect_size)
  ) %>%
  select(bundle, iqm = qc_metric, age_with_qc = age_effect_size)

row3_df <- no_qc %>%
  inner_join(with_qc, by = "bundle") %>%
  filter(iqm %in% iqms_ge_grid)

rho_df <- row3_df %>%
  group_by(iqm) %>%
  summarise(
    rho = suppressWarnings(cor(age_no_qc, age_with_qc, method = "spearman", use = "pairwise.complete.obs")),
    .groups = "drop"
  )

rho_non_contrast <- rho_df %>% filter(iqm != "t1post_dwi_contrast") %>% pull(rho)
rho_non_contrast_min <- min(rho_non_contrast, na.rm = TRUE)
rho_non_contrast_max <- max(rho_non_contrast, na.rm = TRUE)
rho_contrast <- rho_df %>% filter(iqm == "t1post_dwi_contrast") %>% pull(rho)

cat("GE GAM age-IQM stats:
")
print(ge_gam_stats %>% mutate(across(c(r2_age_only, prop_mediated_gam, r2_batch, r2_age_given_batch), ~ round(.x, 4))))
cat("\nRho values:
")
print(rho_df %>% mutate(rho = round(rho, 3)))


In [ ]:
# Final paragraph
r_contrast_txt <- fmt_pct1(r_contrast)
med_contrast_txt <- ifelse(is.finite(med_contrast) && (100 * med_contrast) < 0.01, "< 0.01", fmt_pct1(med_contrast))

txt <- paste0(
  "Several IQMs, including mean framewise displacement (R2 = ", fmt_pct1(r_mean_fd), "%), the quality classifier score (R2 = ", fmt_pct1(r_qc), "%), and preprocessed NDC (R2 = ", fmt_pct1(r_ndc), "%), showed apparent associations with age (Fig. 7a,b). ",
  "However, variance decomposition analyses revealed that these age-IQM relationships were predominantly explained by between-batch differences rather than within-batch participant variation (>= ", fmt_pct1(med_non_contrast_min), "% of the age-IQM relationship explained by batch). ",
  "We next evaluated how including these IQMs as model covariates influenced estimates of developmental effects in white matter microstructure. Bundle-wise FA age effects were estimated in the harmonized data with and without each IQM included as a covariate. ",
  "For all IQMs besides dMRI contrast, adding the metric systemically reduced age effects across bundles (Fig. 7c), consistent with partial absorption of developmental variance through the collinearity between age and acquisition batch. ",
  "The magnitude of attenuation varied across bundles, resulting in a partially reorganized pattern of developmental effects when compared to not covarying for image quality (rho = ", sprintf("%.2f", rho_non_contrast_min), "-", sprintf("%.2f", rho_non_contrast_max), "). ",
  "Notably, preprocessed dMRI contrast showed minimal association with age or batch (R2 = ", r_contrast_txt, "%, ", med_contrast_txt, "% of the age-IQM relationship explained by batch). ",
  "Correspondingly, developmental effects were neither substantially reduced nor reorganized (rho = ", sprintf("%.2f", rho_contrast), "). ",
  "Together, these findings indicate that, in ABCD, many commonly used diffusion IQMs may reflect differences in acquisition rather than participant-level variability. Including such IQMs as model covariates can inadvertently distort biologically meaningful developmental signals."
)

cat(txt, "\n")
